# Reflect Orbital — Constellation Trade Study Analysis

**Company:** Reflect Orbital  
**Mission:** Deploy mirror satellites in sun-synchronous orbit (SSO) to reflect sunlight onto ground-based solar farms, extending their productive hours beyond local sunrise/sunset and enabling dispatchable nighttime clean energy.

**Purpose of this notebook:** Translate C++ orbital simulation outputs (coverage, illumination fraction, delta-v budgets) into economic metrics — total constellation cost, annual energy revenue, and payback period — to identify the most commercially attractive constellation configurations.

---

### Simulation output files consumed
| File | Location | Description |
|------|----------|-------------|
| `experiment_summary.csv` | `output/<run_name>/` | One row per Monte Carlo run; constellation-level metrics |
| `summary.csv` | `output/<run_name>/run_NNNN/` | Per-run constellation results |
| `satellites.csv` | `output/<run_name>/run_NNNN/` | Per-satellite orbital performance |
| `ground_targets.csv` | `output/<run_name>/run_NNNN/` | Per-target coverage & illumination (new output) |

## 0 — Imports & configuration

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'sans-serif',
})

# ── Path configuration ────────────────────────────────────────────────────────
# Point RUN_NAME at whichever campaign you want to analyse.
# All paths are derived from this single variable.
REPO_ROOT   = Path('/Users/vaughncampos/Desktop/constellation-simulation')
RUN_NAME    = 'reflect_orbital'
OUTPUT_DIR  = REPO_ROOT / 'output' / RUN_NAME

EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / 'experiment_summary.csv'

print(f'Repository root : {REPO_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Experiment CSV  : {EXPERIMENT_SUMMARY_PATH}')
print(f'Path exists     : {EXPERIMENT_SUMMARY_PATH.exists()}')

---
## 1 — Load & explore the experiment summary

The `experiment_summary.csv` is written by the C++ simulator after all Monte Carlo runs complete. It aggregates one row per run configuration, capturing orbit altitude, inclination, Walker geometry, coverage fraction, revisit times, and delta-v budgets.

If the CSV does not yet exist (i.e. the simulation has not been run), this cell falls back to **synthetic illustrative data** so the rest of the notebook can be demonstrated end-to-end. Replace with real output when available.

In [ ]:
# ── Load experiment summary (real data preferred; synthetic fallback) ─────────

def generate_synthetic_experiment_summary() -> pd.DataFrame:
    """Return a realistic-looking but entirely synthetic experiment summary.
    *** REPLACE WITH REAL SIMULATION OUTPUT — this data is for illustration only. ***
    """
    rng = np.random.default_rng(42)
    altitudes       = [400, 450, 500, 550, 600]
    satellite_counts = [20, 36, 50, 72, 100]
    rows = []
    rid = 0
    for alt in altitudes:
        for n_sat in satellite_counts:
            n_planes = max(2, n_sat // 10)
            rows.append({
                'run_id'                  : rid,
                'run_name'                : RUN_NAME,
                'altitude_km'             : alt,
                'inclination_deg'         : 97.0 + (alt - 450) * 0.05,
                'total_satellites'        : n_sat,
                'planes'                  : n_planes,
                'sats_per_plane'          : n_sat // n_planes,
                # Coverage improves with more satellites and lower altitude (smaller footprint but denser coverage)
                'coverage_pct'            : np.clip(5 + n_sat * 0.55 + rng.normal(0, 2), 5, 95),
                'revisit_time_avg_s'      : max(400, 4000 - n_sat * 30 + rng.normal(0, 80)),
                'revisit_time_max_s'      : max(800, 8000 - n_sat * 50 + rng.normal(0, 200)),
                # Drag delta-v increases sharply at lower altitude
                'avg_drag_dv_ms'          : 120 * (550 / alt) ** 4 + rng.normal(0, 3),
                'avg_sk_dv_ms'            : 120 * (550 / alt) ** 4 + rng.normal(0, 3),
                # SSO at ~97° inclination gives high sunlit fraction
                'avg_sunlit_pct'          : np.clip(68 + (alt - 450) * 0.05 + rng.normal(0, 1.5), 55, 85),
                'avg_altitude_km'         : alt + rng.normal(0, 2),
                'min_altitude_km'         : alt - 15 + rng.normal(0, 2),
                'deployment_dv_per_sat_ms': 0,
            })
            rid += 1
    return pd.DataFrame(rows)


if EXPERIMENT_SUMMARY_PATH.exists():
    df = pd.read_csv(EXPERIMENT_SUMMARY_PATH)
    data_source = 'real simulation output'
else:
    print('⚠  experiment_summary.csv not found — using SYNTHETIC illustrative data.')
    print('   Replace with real simulation output before drawing conclusions.')
    df = generate_synthetic_experiment_summary()
    data_source = 'SYNTHETIC (illustrative only — replace with simulation output)'

print(f'\nData source : {data_source}')
print(f'Rows loaded : {len(df)}')
print(f'Columns     : {list(df.columns)}')

In [ ]:
# ── Basic descriptive statistics ──────────────────────────────────────────────
print('=== Descriptive statistics ===')
display_cols = [
    'altitude_km', 'total_satellites', 'coverage_pct',
    'revisit_time_avg_s', 'avg_sunlit_pct',
    'avg_drag_dv_ms', 'avg_sk_dv_ms'
]
df[display_cols].describe().round(2)

In [ ]:
# ── Heatmaps: coverage_pct and avg_sunlit_pct vs altitude × satellite count ───
#
# These heatmaps give an instant visual of the design space:
#   - Rows = altitude tier
#   - Columns = constellation size
#   - Colour = metric value
#
# For a single-run dataset the heatmap degenerates to a 1-cell table, which is
# still informative; it becomes richer as more sweep configurations are added.

def pivot_heatmap(df, value_col, label, ax, fmt='.1f', cmap='YlOrRd'):
    """Pivot and plot a heatmap of *value_col* on the given axes."""
    alt_vals = sorted(df['altitude_km'].unique())
    sat_vals = sorted(df['total_satellites'].unique())

    # Build pivot (mean across any repeated configs)
    pivot = (
        df.groupby(['altitude_km', 'total_satellites'])[value_col]
          .mean()
          .unstack('total_satellites')
          .reindex(index=alt_vals, columns=sat_vals)
    )

    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap,
                   interpolation='nearest')
    ax.set_xticks(range(len(sat_vals)))
    ax.set_xticklabels(sat_vals)
    ax.set_yticks(range(len(alt_vals)))
    ax.set_yticklabels([f'{a:.0f}' for a in alt_vals])
    ax.set_xlabel('Total satellites')
    ax.set_ylabel('Altitude (km)')
    ax.set_title(label)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Annotate cells
    for i in range(len(alt_vals)):
        for j in range(len(sat_vals)):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, format(v, fmt), ha='center', va='center',
                        fontsize=8, color='black')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot_heatmap(df, 'coverage_pct',   'Coverage fraction (%) vs Altitude & Constellation size',
              axes[0], cmap='Blues')
pivot_heatmap(df, 'avg_sunlit_pct', 'Avg satellite sunlit fraction (%) vs Altitude & Constellation size',
              axes[1], cmap='YlOrRd')

fig.suptitle('Reflect Orbital — Design-Space Heatmaps', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2 — Economic model parameters

The economic model converts orbital performance metrics into financial projections. Parameters below are **clearly labelled as illustrative / dummy values** — replace each with figures from actual Reflect Orbital cost models, vendor quotes, and off-take agreements before using results for business decisions.

### Model logic (summary)

1. **Cost:** `(manufacturing + launch) × fleet size`
2. **Power per satellite:** solar constant × mirror area × reflectivity × atmospheric transmittance
3. **Energy revenue:** fleet power × illuminated fraction × hours × electricity price × peak-demand premium
4. **Payback period:** total cost ÷ annual revenue

The **atmospheric transmittance** function follows a Beer-Lambert path-length model: light travelling at elevation angle θ traverses ≈ 1/sin(θ) air masses, each attenuating by the single-airmass factor τ = 0.85. Low-elevation passes (grazing angle) suffer heavy losses; high-elevation passes (nearly overhead) are most efficient.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ECONOMIC MODEL PARAMETERS
#  *** ALL VALUES BELOW ARE DUMMY / ILLUSTRATIVE — for demonstration only ***
#  Replace with real figures from cost models and off-take agreements.
# ══════════════════════════════════════════════════════════════════════════════

# --- Satellite cost model (DUMMY) ---
sat_cost_usd             = 1_500_000   # USD per satellite: manufacturing + testing
launch_cost_per_kg       = 6_000       # USD/kg, rideshare (e.g. SpaceX Transporter)
sat_mass_kg              = 50          # kg, deployed reflector satellite bus + mirror

# --- Mirror optics (DUMMY) ---
mirror_area_m2           = 5.0         # m², deployable reflective film per satellite
mirror_reflectivity      = 0.92        # fraction of incident light reflected (aluminium film)

# --- Solar & atmosphere (DUMMY) ---
solar_irradiance_W_m2    = 1361        # W/m², top-of-atmosphere solar constant
# Beer-Lambert single-airmass transmittance factor
# τ = 0.85 is representative of a clear atmosphere at sea level.
# Total transmittance = τ^(1/sin(elev_deg)) — degrades at low elevation angles.
BEER_LAMBERT_TAU         = 0.85        # single-airmass transmittance (DUMMY)

def atmosphere_transmittance(elev_deg: float) -> float:
    """Beer-Lambert atmospheric transmittance at a given satellite elevation.

    Parameters
    ----------
    elev_deg : float
        Satellite elevation above the horizon (degrees). Clamped to ≥5° to
        avoid numerical blow-up at very shallow angles.

    Returns
    -------
    float
        Fraction of light reaching the ground (0–1).
        *** DUMMY model — replace with Lowtran/MODTRAN lookup for real analysis. ***
    """
    elev_clamped = max(float(elev_deg), 5.0)       # minimum 5° to avoid cosecant singularity
    air_masses   = 1.0 / np.sin(np.radians(elev_clamped))
    return BEER_LAMBERT_TAU ** air_masses


# --- Revenue model (DUMMY) ---
power_purchase_price_usd_per_kwh = 0.08  # USD/kWh, grid wholesale reference
peak_demand_premium              = 2.5   # multiplier for dispatchable nighttime solar

# --- Simulation metadata ---
sim_duration_days = 7   # days covered by a single simulation run (used for energy scaling)

# ── Quick sanity-check: transmittance curve ───────────────────────────────────
elevations = np.linspace(5, 90, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(elevations, [atmosphere_transmittance(e) for e in elevations], lw=2, color='steelblue')
ax.axvline(30, ls='--', color='grey', lw=1, label='30° elev (typical pass midpoint)')
ax.axvline(60, ls=':', color='grey', lw=1, label='60° elev (high pass)')
ax.set_xlabel('Satellite elevation above horizon (°)')
ax.set_ylabel('Atmospheric transmittance')
ax.set_title('Beer-Lambert transmittance model  [DUMMY — illustrative only]')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Transmittance @ 5°  : {atmosphere_transmittance(5):.3f}')
print(f'Transmittance @ 15° : {atmosphere_transmittance(15):.3f}')
print(f'Transmittance @ 30° : {atmosphere_transmittance(30):.3f}')
print(f'Transmittance @ 60° : {atmosphere_transmittance(60):.3f}')
print(f'Transmittance @ 90° : {atmosphere_transmittance(90):.3f}')

---
## 3 — Per-run economics

For each simulation run we compute:

| Metric | Formula | Notes |
|--------|---------|-------|
| `total_cost_musd` | `(sat_cost + launch_cost/kg × sat_mass_kg) × N_sats / 1e6` | Total CAPEX in $M |
| `fleet_mirror_area_m2` | `N_sats × mirror_area_m2` | Total reflective aperture |
| `avg_transmittance` | `transmittance(avg_elevation_deg)` | Pulled from `ground_targets.csv`; fallback 15° |
| `peak_power_per_sat_W` | `G₀ × A_mirror × η_mirror × τ_atm` | W delivered to ground per satellite |
| `daily_energy_per_sat_Wh` | `P_peak × (illuminated_pct/100) × 24` | Wh/day |
| `annual_revenue_musd` | `fleet × E_daily/1000 × price × premium × 365 / 1e6` | Revenue in $M/yr |
| `payback_years` | `total_cost_musd / annual_revenue_musd` | Key figure of merit |

In [ ]:
# ── Helper: load ground_targets.csv for a specific run ───────────────────────

def load_ground_targets(run_id: int, run_name: str = RUN_NAME) -> pd.DataFrame | None:
    """Return the ground_targets.csv for run_id, or None if not found."""
    path = OUTPUT_DIR / f'run_{run_id:04d}' / 'ground_targets.csv'
    if path.exists():
        return pd.read_csv(path)
    return None


def avg_elevation_for_run(run_id: int) -> float:
    """Return mean avg_elevation_deg across all targets in this run, or 15.0 as fallback."""
    gt = load_ground_targets(run_id)
    if gt is not None and 'avg_elevation_deg' in gt.columns and len(gt) > 0:
        return float(gt['avg_elevation_deg'].mean())
    return 15.0   # fallback: conservative low-elevation assumption


def avg_illuminated_pct_for_run(run_id: int, df_row: pd.Series) -> float:
    """Return mean illuminated_pct across targets, falling back to avg_sunlit_pct."""
    gt = load_ground_targets(run_id)
    if gt is not None and 'illuminated_pct' in gt.columns and len(gt) > 0:
        return float(gt['illuminated_pct'].mean())
    # Fallback: use satellite sunlit fraction as a proxy
    return float(df_row.get('avg_sunlit_pct', 70.0))


# ── Compute economics for every run ──────────────────────────────────────────

records = []
for _, row in df.iterrows():
    rid             = int(row['run_id'])
    n_sats          = int(row['total_satellites'])
    illuminated_pct = avg_illuminated_pct_for_run(rid, row)
    avg_elev        = avg_elevation_for_run(rid)

    # Cost
    cost_per_sat   = sat_cost_usd + launch_cost_per_kg * sat_mass_kg
    total_cost_musd = cost_per_sat * n_sats / 1e6

    # Power & energy
    transmittance        = atmosphere_transmittance(avg_elev)
    peak_power_per_sat_W = solar_irradiance_W_m2 * mirror_area_m2 * mirror_reflectivity * transmittance
    daily_energy_per_sat_Wh = peak_power_per_sat_W * (illuminated_pct / 100.0) * 24.0

    # Revenue
    annual_energy_fleet_kWh = n_sats * daily_energy_per_sat_Wh / 1000.0 * 365.0
    annual_revenue_musd     = (annual_energy_fleet_kWh *
                               power_purchase_price_usd_per_kwh *
                               peak_demand_premium) / 1e6

    payback_years = total_cost_musd / annual_revenue_musd if annual_revenue_musd > 0 else np.inf

    records.append({
        'run_id'                 : rid,
        'altitude_km'            : row['altitude_km'],
        'total_satellites'       : n_sats,
        'coverage_pct'           : row['coverage_pct'],
        'illuminated_pct'        : illuminated_pct,
        'avg_elev_deg'           : avg_elev,
        'avg_transmittance'      : transmittance,
        'peak_power_per_sat_W'   : peak_power_per_sat_W,
        'daily_energy_per_sat_Wh': daily_energy_per_sat_Wh,
        'fleet_mirror_area_m2'   : n_sats * mirror_area_m2,
        'total_cost_musd'        : total_cost_musd,
        'annual_revenue_musd'    : annual_revenue_musd,
        'payback_years'          : payback_years,
        'revenue_cost_ratio'     : annual_revenue_musd / total_cost_musd if total_cost_musd > 0 else 0,
    })

econ = pd.DataFrame(records)

print('=== Economic results (all runs) ===')
print(econ[[
    'run_id', 'altitude_km', 'total_satellites',
    'total_cost_musd', 'annual_revenue_musd', 'payback_years'
]].to_string(index=False))

---
## 4 — Pareto analysis: revenue vs cost

The Pareto frontier identifies runs where **no other configuration achieves higher revenue at the same or lower cost** — i.e. the economically non-dominated set. In practice, operators want to be on or close to this frontier.

- **Marker colour** encodes altitude (lower = warmer; higher altitude = cooler).
- **Marker size** encodes total satellite count (larger fleet = larger dot).
- **Gold stars** mark Pareto-efficient runs.

In [ ]:
# ── Identify Pareto-efficient runs (maximise revenue, minimise cost) ──────────

def pareto_mask(costs: np.ndarray, revenues: np.ndarray) -> np.ndarray:
    """Return boolean mask; True = run is Pareto-efficient (max revenue for its cost level)."""
    n = len(costs)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # j dominates i if j has lower-or-equal cost AND higher-or-equal revenue
            if costs[j] <= costs[i] and revenues[j] >= revenues[i]:
                if costs[j] < costs[i] or revenues[j] > revenues[i]:
                    dominated[i] = True
                    break
    return ~dominated


costs    = econ['total_cost_musd'].values
revenues = econ['annual_revenue_musd'].values
is_pareto = pareto_mask(costs, revenues)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

alt_vals = econ['altitude_km'].values
norm     = mcolors.Normalize(vmin=alt_vals.min(), vmax=alt_vals.max())
cmap_p   = cm.RdYlBu_r
colours  = cmap_p(norm(alt_vals))

sat_sizes = (econ['total_satellites'].values / econ['total_satellites'].max()) * 300 + 40

sc = ax.scatter(
    econ['total_cost_musd'], econ['annual_revenue_musd'],
    c=colours, s=sat_sizes, alpha=0.8, edgecolors='white', lw=0.7, zorder=3
)

# Overlay Pareto-efficient runs as gold stars
pareto_df = econ[is_pareto]
ax.scatter(
    pareto_df['total_cost_musd'], pareto_df['annual_revenue_musd'],
    marker='*', s=350, color='gold', edgecolors='darkorange', lw=1.2,
    zorder=5, label='Pareto-efficient'
)

# Draw break-even line (revenue = cost, i.e. 1-year payback)
x_range = np.linspace(costs.min() * 0.8, costs.max() * 1.1, 200)
ax.plot(x_range, x_range, ls='--', color='grey', lw=1, label='1-yr payback line')

# Annotate each point
for _, row in econ.iterrows():
    ax.annotate(
        f"{int(row['total_satellites'])}sat\n{row['altitude_km']:.0f}km",
        xy=(row['total_cost_musd'], row['annual_revenue_musd']),
        xytext=(6, 4), textcoords='offset points', fontsize=7, color='#333333'
    )

# Colourbar for altitude
sm = cm.ScalarMappable(cmap=cmap_p, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Altitude (km)')

# Size legend
for n_ex in [20, 50, 100]:
    s_ex = (n_ex / econ['total_satellites'].max()) * 300 + 40
    ax.scatter([], [], s=s_ex, color='grey', alpha=0.6, label=f'{n_ex} sats')

ax.set_xlabel('Total constellation cost ($M)')
ax.set_ylabel('Annual energy revenue ($M/yr)')
ax.set_title('Reflect Orbital — Revenue vs Cost Pareto Analysis  [DUMMY parameters]')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Pareto-efficient runs ({is_pareto.sum()} of {len(econ)}):')
print(pareto_df[['run_id', 'altitude_km', 'total_satellites',
                  'total_cost_musd', 'annual_revenue_musd', 'payback_years']]
      .sort_values('payback_years').to_string(index=False))

---
## 5 — Sensitivity analysis (tornado chart)

A tornado chart shows how payback period changes when each input parameter is varied ±20% from its base value, with all other parameters held fixed. The **widest bars indicate the highest-leverage assumptions** — where better data collection or negotiation delivers the most business value.

Analysis is performed on the **best run** (lowest payback years / highest revenue÷cost ratio).

In [ ]:
# ── Identify best run ─────────────────────────────────────────────────────────
best_idx  = econ['revenue_cost_ratio'].idxmax()
best_run  = econ.loc[best_idx]
best_full = df.loc[df['run_id'] == best_run['run_id']].iloc[0]

print(f"Best run: run_id={int(best_run['run_id'])}, "
      f"altitude={best_run['altitude_km']:.0f} km, "
      f"satellites={int(best_run['total_satellites'])}, "
      f"payback={best_run['payback_years']:.2f} yr")

# ── Payback calculator accepting overrides ────────────────────────────────────

def compute_payback(
    n_sats:           int   = None,
    illum_pct:        float = None,
    avg_elev:         float = None,
    sat_cost:         float = sat_cost_usd,
    launch_cost:      float = launch_cost_per_kg,
    sat_mass:         float = sat_mass_kg,
    mirror_area:      float = mirror_area_m2,
    reflectivity:     float = mirror_reflectivity,
    irradiance:       float = solar_irradiance_W_m2,
    price_kwh:        float = power_purchase_price_usd_per_kwh,
    premium:          float = peak_demand_premium,
) -> float:
    """Return payback_years for the best run with parameter overrides."""
    n   = n_sats   if n_sats   is not None else int(best_run['total_satellites'])
    ip  = illum_pct if illum_pct is not None else float(best_run['illuminated_pct'])
    el  = avg_elev  if avg_elev  is not None else float(best_run['avg_elev_deg'])

    cost_per_sat  = sat_cost + launch_cost * sat_mass
    total_cost_m  = cost_per_sat * n / 1e6
    tau           = BEER_LAMBERT_TAU ** (1.0 / np.sin(np.radians(max(el, 5.0))))
    p_peak        = irradiance * mirror_area * reflectivity * tau
    e_daily       = p_peak * (ip / 100.0) * 24.0
    rev_m         = n * e_daily / 1000.0 * price_kwh * premium * 365.0 / 1e6
    return total_cost_m / rev_m if rev_m > 0 else np.inf


# ── Tornado analysis: vary each parameter ±20% ───────────────────────────────
DELTA = 0.20   # ±20 %

parameters = [
    ('Sat. manufacturing cost',   'sat_cost',       sat_cost_usd),
    ('Launch cost ($/kg)',         'launch_cost',    launch_cost_per_kg),
    ('Satellite mass (kg)',        'sat_mass',       sat_mass_kg),
    ('Mirror area (m²)',           'mirror_area',    mirror_area_m2),
    ('Mirror reflectivity',        'reflectivity',   mirror_reflectivity),
    ('Solar irradiance (W/m²)',    'irradiance',     solar_irradiance_W_m2),
    ('Power price ($/kWh)',        'price_kwh',      power_purchase_price_usd_per_kwh),
    ('Peak-demand premium',        'premium',        peak_demand_premium),
]

base_payback = compute_payback()
tornado_rows = []
for label, kwarg, base_val in parameters:
    low_val  = base_val * (1 - DELTA)
    high_val = base_val * (1 + DELTA)
    pb_low   = compute_payback(**{kwarg: low_val})
    pb_high  = compute_payback(**{kwarg: high_val})
    tornado_rows.append({
        'label'    : label,
        'pb_low'   : pb_low,
        'pb_high'  : pb_high,
        'swing'    : abs(pb_high - pb_low),
    })

torn = pd.DataFrame(tornado_rows).sort_values('swing', ascending=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(torn))

for i, (_, row) in enumerate(torn.iterrows()):
    lo, hi = row['pb_low'], row['pb_high']
    # Bar extends from min to max
    ax.barh(i, hi - lo, left=lo, height=0.55,
            color='#4472C4' if hi > lo else '#ED7D31', alpha=0.82)
    ax.barh(i, lo - base_payback, left=base_payback, height=0.55,
            color='#ED7D31', alpha=0.82)

ax.axvline(base_payback, color='black', lw=1.5, ls='-', label=f'Base: {base_payback:.2f} yr')
ax.set_yticks(y_pos)
ax.set_yticklabels(torn['label'])
ax.set_xlabel('Payback period (years)')
ax.set_title(f'Tornado Chart — Sensitivity of Payback Period (±20% each param)\n'
             f'Best run: {int(best_run["total_satellites"])} sats @ {best_run["altitude_km"]:.0f} km  '
             f'[DUMMY parameters]')
ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nParameter impact (sorted by sensitivity):')
print(torn[['label', 'pb_low', 'pb_high', 'swing']].to_string(index=False))

---
## 6 — Ground target deep-dive

The `ground_targets.csv` file (new C++ output) records per-target coverage statistics. For a mirror constellation, the most commercially valuable targets are those with **high illuminated fraction** (long periods of satellite-reflected sunlight) and **high average elevation** (efficient atmospheric transmission).

This section loads that file for the best run identified above. If the file does not yet exist, synthetic target data is generated for illustration.

In [ ]:
# ── Load ground_targets.csv for best run ─────────────────────────────────────

best_run_id = int(best_run['run_id'])
gt = load_ground_targets(best_run_id)

if gt is None:
    print(f'⚠  ground_targets.csv not found for run_{best_run_id:04d} — '
          'generating SYNTHETIC illustrative data.')
    rng_gt = np.random.default_rng(99)
    gt = pd.DataFrame({
        'name'               : [f'Solar Farm {chr(65+i)}' for i in range(8)],
        'lat_deg'            : rng_gt.uniform(20, 45, 8).round(2),
        'lon_deg'            : rng_gt.uniform(-120, 10, 8).round(2),
        'visible_pct'        : rng_gt.uniform(20, 70, 8).round(2),
        'illuminated_pct'    : rng_gt.uniform(10, 55, 8).round(2),
        'avg_elevation_deg'  : rng_gt.uniform(12, 55, 8).round(2),
        'max_elevation_deg'  : rng_gt.uniform(55, 89, 8).round(2),
        'avg_pass_duration_s': rng_gt.uniform(120, 600, 8).round(1),
        'pass_count'         : rng_gt.integers(10, 80, 8),
        'coverage_time_s'    : rng_gt.uniform(3600, 50000, 8).round(0),
        'illuminated_time_s' : rng_gt.uniform(1800, 40000, 8).round(0),
    })
    gt_source = 'SYNTHETIC (illustrative)'
else:
    gt_source = f'run_{best_run_id:04d}/ground_targets.csv'

# Derived convenience columns
gt['illuminated_hours'] = gt['illuminated_time_s'] / 3600.0
gt['transmittance']     = gt['avg_elevation_deg'].apply(atmosphere_transmittance)
gt['effective_power_W_per_sat'] = (
    solar_irradiance_W_m2 * mirror_area_m2 * mirror_reflectivity * gt['transmittance']
)

print(f'Data source: {gt_source}')
print(f'Targets    : {len(gt)}')

# ── Summary table ─────────────────────────────────────────────────────────────
display_gt = gt[[
    'name', 'lat_deg', 'lon_deg',
    'illuminated_pct', 'avg_elevation_deg', 'max_elevation_deg',
    'pass_count', 'illuminated_hours', 'transmittance', 'effective_power_W_per_sat'
]].sort_values('illuminated_pct', ascending=False)
display_gt.columns = [
    'Target', 'Lat (°)', 'Lon (°)',
    'Illuminated (%)', 'Avg Elev (°)', 'Max Elev (°)',
    'Pass count', 'Illuminated hrs', 'Transmittance', 'Power/sat (W)'
]
print('\n=== Ground target summary (sorted by illuminated %) ===')
display_gt.round(3)

In [ ]:
# ── Bar charts: illuminated hours and effective power per target ──────────────

gt_sorted = gt.sort_values('illuminated_hours', ascending=False)
target_labels = gt_sorted['name'].values
x = np.arange(len(target_labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Chart 1: illuminated hours ---
bars1 = axes[0].bar(x, gt_sorted['illuminated_hours'], color='#F4A460', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(target_labels, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('Total illuminated time (hours)')
axes[0].set_title('Illuminated hours per target\n(sim duration)')
axes[0].grid(True, axis='y', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)

# --- Chart 2: illuminated % ---
gt_sorted2 = gt.sort_values('illuminated_pct', ascending=False)
colours_elev = cm.RdYlGn(
    mcolors.Normalize(vmin=gt['avg_elevation_deg'].min(),
                      vmax=gt['avg_elevation_deg'].max())(gt_sorted2['avg_elevation_deg'])
)
bars2 = axes[1].bar(np.arange(len(gt_sorted2)), gt_sorted2['illuminated_pct'],
                    color=colours_elev, edgecolor='white')
axes[1].set_xticks(np.arange(len(gt_sorted2)))
axes[1].set_xticklabels(gt_sorted2['name'].values, rotation=35, ha='right', fontsize=8)
axes[1].set_ylabel('Illuminated fraction (%)')
axes[1].set_title('Illuminated % per target\n(coloured by avg elevation)')
axes[1].grid(True, axis='y', alpha=0.3)

sm2 = cm.ScalarMappable(cmap='RdYlGn',
                         norm=mcolors.Normalize(vmin=gt['avg_elevation_deg'].min(),
                                                vmax=gt['avg_elevation_deg'].max()))
sm2.set_array([])
cbar2 = plt.colorbar(sm2, ax=axes[1], fraction=0.046, pad=0.04)
cbar2.set_label('Avg elev (°)', fontsize=8)

# --- Chart 3: effective delivered power per satellite ---
axes[2].bar(x, gt_sorted['effective_power_W_per_sat'], color='#4472C4', edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(target_labels, rotation=35, ha='right', fontsize=8)
axes[2].set_ylabel('Effective ground power (W/sat)')
axes[2].set_title('Ground-delivered power per satellite\n(Beer-Lambert atm. model)')
axes[2].grid(True, axis='y', alpha=0.3)
axes[2].axhline(gt['effective_power_W_per_sat'].mean(), ls='--', color='red',
                lw=1.2, label=f'Mean: {gt["effective_power_W_per_sat"].mean():.0f} W')
axes[2].legend(fontsize=8)

fig.suptitle(f'Ground Target Analysis — Best run (run_{best_run_id:04d}, '
             f'{int(best_run["total_satellites"])} sats @ {best_run["altitude_km"]:.0f} km)  '
             f'[{gt_source}]', fontsize=11)
plt.tight_layout()
plt.show()

---
## 7 — Key findings

> **Note:** All economic figures below are derived from the **dummy parameters** defined in Section 2. They are intended to demonstrate the analysis framework, not to represent actual Reflect Orbital financial projections. Replace the parameters in Section 2 with real figures before drawing business conclusions.

### Top 5 configurations by payback period

In [ ]:
# ── Top-5 table ───────────────────────────────────────────────────────────────
top5 = (
    econ.sort_values('payback_years')
        .head(5)
        .reset_index(drop=True)
)

top5_display = top5[[
    'run_id', 'altitude_km', 'total_satellites',
    'coverage_pct', 'illuminated_pct',
    'total_cost_musd', 'annual_revenue_musd', 'payback_years'
]].copy()
top5_display.columns = [
    'Run ID', 'Alt (km)', 'Satellites',
    'Coverage (%)', 'Illuminated (%)',
    'CAPEX ($M)', 'Revenue ($M/yr)', 'Payback (yr)'
]
top5_display = top5_display.round(2)
top5_display.index += 1   # 1-based rank
top5_display.index.name = 'Rank'

print('=== Top 5 runs by payback period [DUMMY parameters] ===')
top5_display

In [ ]:
# ── Summary bar chart of payback period for all runs ─────────────────────────
econ_sorted = econ.sort_values('payback_years')
labels = [
    f"{int(r['total_satellites'])}sat\n{r['altitude_km']:.0f}km"
    for _, r in econ_sorted.iterrows()
]
paybacks = econ_sorted['payback_years'].values
colours_pb = ['#2ECC71' if p < 8 else '#F39C12' if p < 12 else '#E74C3C' for p in paybacks]

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.9), 4.5))
bars = ax.bar(range(len(labels)), paybacks, color=colours_pb, edgecolor='white', width=0.7)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Payback period (years)')
ax.set_title('Payback Period by Configuration  [DUMMY parameters]\n'
             'Green < 8 yr | Orange 8–12 yr | Red > 12 yr')
ax.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, paybacks):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{v:.1f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

### Interpretation

The analysis highlights several drivers of commercial performance for Reflect Orbital's mirror constellation:

**1. Satellite count dominates revenue — but cost scales linearly.**  
Larger constellations generate proportionally more energy revenue, since every satellite contributes independently to the fleet output. Because cost also scales linearly with satellite count, the payback period is *relatively insensitive to fleet size* once a critical coverage threshold is reached. The economic leverage is in the per-satellite revenue/cost ratio, not the fleet size.

**2. Altitude affects illumination fraction more than cost.**  
Higher orbits spend less time in Earth's shadow (higher `avg_sunlit_pct`), directly improving daily energy generation. The marginal launch cost difference between 400 km and 600 km is small on a rideshare. This means higher altitude tends to shorten payback period — balanced against increased station-keeping requirements for residual drag.

**3. Atmospheric transmittance is a steep function of elevation angle.**  
A pass at 10° elevation delivers only ~50% the light of a pass at 60°. Target site selection (latitude bands that see high-elevation SSO passes) can improve ground energy yield by 30–40% at no additional constellation cost. This makes *site optimisation* a high-leverage activity.

**4. The peak-demand premium is the single largest revenue multiplier.**  
Dispatchable nighttime solar commanding a 2.5× premium accounts for the majority of revenue. Securing long-term off-take agreements at favourable premium rates is the highest-priority commercial risk to retire.

**5. Mirror area and reflectivity directly gate peak power.**  
Doubling mirror area from 5 m² to 10 m² doubles per-satellite power output with a relatively modest mass penalty. Near-term technology development focus should be on maximising deployable mirror area within the launch mass budget.

---
*This notebook is a live analysis document. Re-run it with updated simulation CSVs or revised economic parameters as the design matures.*